# 07 - WRF 1-hour peak intensity (phase 2 sub-daily metric)

Turns the WRF hourly precipitation subsets (notebook 02) into the `max_1hr_precip`
metric: the mean water-year (Sep-Aug, matching the WRF run boundaries) maximum
1-hour rate, per 30-year window.

1. Per model (the 3-model `precip_hourly` subset): stitch historical + ssp370,
   water-year annual maxima, then window means for the baseline and each horizon.
2. **Change factors are computed per model** (horizon / that model's own baseline)
   before ensembling - the standard bias-robust form for presenting WRF intensities.
3. Ensemble median / p10 / p90 across models (n=3: the percentiles are close to the
   envelope; label accordingly).
4. Clip to the buffered TRPA boundary; write NetCDF (native curvilinear 9 km grid),
   GeoTIFF (nearest-resampled to a regular grid, EPSG:26910), a summary CSV, and
   `outputs/wrf_i1h_grid.geojson` - the cell-polygon payload the storm events
   explorer (`html/tahoe-precip-events.html`) embeds.

Scenario asymmetry: **ssp370 only** (no WRF ssp245 runs exist). Publication of the
GeoJSON into the public page is a deliberate manual-review step, same as the
projections payload from notebook 05.

In [1]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("07_wrf_hourly_intensity")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]

MODELS = cfg["sources"]["caladapt"]["wrf"]["precip_hourly"]["models"]
HORIZONS = cfg["analysis"]["horizons"]
BASE = cfg["analysis"]["baseline"]
SCEN = "ssp370"   # only WRF scenario that exists
log.info(f"models: {MODELS}; baseline WY {BASE[0]}-{BASE[1]}; "
         f"horizons: {[h['name'] for h in HORIZONS]}; scenario: {SCEN}")

Python: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe


2026-08-17 16:34:15 | INFO | 07_wrf_hourly_intensity | Log file: C:\Users\mbindl\Documents\GitHub\PROTECT\climate\logs\07_wrf_hourly_intensity_2026-08-17_163415.log


2026-08-17 16:34:15 | INFO | 07_wrf_hourly_intensity | models: ['access-cm2', 'ec-earth3', 'miroc6']; baseline WY 1981-2010; horizons: ['2020-2049', '2040-2069', '2070-2099']; scenario: ssp370


## Boundary + buffer (same as notebook 05)

In [2]:
import geopandas as gpd
import requests as _rq

bnd_json = _rq.get(f"{cfg['study_area']['boundary_url']}/query",
                   params={"where": "1=1", "outFields": "OBJECTID", "f": "geojson"},
                   timeout=120).json()
boundary = gpd.GeoDataFrame.from_features(bnd_json["features"], crs="EPSG:4326")
_utm = boundary.to_crs(epsg=cfg["crs"]["target_epsg"]).buffer(
    cfg["study_area"]["buffer_km"] * 1000)
buffered = _utm.union_all() if hasattr(_utm, "union_all") else _utm.unary_union
buffered_ll = gpd.GeoSeries([buffered], crs=f"EPSG:{cfg['crs']['target_epsg']}") \
    .to_crs("EPSG:4326").iloc[0]
log.info(f"boundary buffered {cfg['study_area']['buffer_km']} km; "
         f"lat/lon bounds: {[round(b, 3) for b in buffered_ll.bounds]}")

2026-08-17 16:34:30 | INFO | 07_wrf_hourly_intensity | boundary buffered 10 km; lat/lon bounds: [-120.368, 38.614, -119.762, 39.415]


## Water-year annual maxima per model
WRF runs are Sep-Aug; `YS-SEP` resampling aligns exactly (no partial years except
one missing hour at the ACCESS-CM2 historical start, from the accumulator diff).
Window `WY a-b` = `a-1`-09-01 through `b`-08-31.

In [3]:
cad_dir = RAW / "caladapt"

def wy_window(da, y0, y1):
    """Water years y0..y1 of an annual (YS-SEP) series: periods starting Sep y0-1..Sep y1-1."""
    return da.sel(time=slice(f"{y0 - 1}-09-01", f"{y1 - 1}-09-02"))

per_model = {}   # model -> {"baseline": DataArray(y,x), horizon_name: ..., "cf_"+hz: ...}
ref = None       # reference dataset for coords/masks
for model in MODELS:
    f_hist = cad_dir / f"wrf_{model}_historical_prec_hourly__tahoe.nc"
    f_ssp = cad_dir / f"wrf_{model}_{SCEN}_prec_hourly__tahoe.nc"
    if not (f_hist.exists() and f_ssp.exists()):
        log.warning(f"[{model}] hourly files missing - skipped")
        continue
    try:
        ds_h = xr.open_dataset(f_hist, chunks={"time": 24 * 365 * 5})
        ds_s = xr.open_dataset(f_ssp, chunks={"time": 24 * 365 * 5})
    except ValueError:   # dask unavailable - load eagerly as float32
        ds_h = xr.open_dataset(f_hist)
        ds_s = xr.open_dataset(f_ssp)
    if ref is None:
        ref = xr.open_dataset(f_hist)
    full = xr.concat([ds_h["prec_mm_hr"].astype("float32"),
                      ds_s["prec_mm_hr"].astype("float32")], dim="time").sortby("time")
    ann = full.resample(time="YS-SEP").max().compute()
    n_hr = int(full.sizes["time"])
    log.info(f"[{model}] {n_hr:,} hourly steps -> {int(ann.sizes['time'])} water-year maxima "
             f"({str(full.time.values[0])[:10]} .. {str(full.time.values[-1])[:10]})")
    out = {"baseline": wy_window(ann, *BASE).mean("time")}
    for hz in HORIZONS:
        wmean = wy_window(ann, hz["start"], hz["end"]).mean("time")
        out[hz["name"]] = wmean
        out["cf_" + hz["name"]] = wmean / out["baseline"]   # per-model change factor
    per_model[model] = out
    ds_h.close(); ds_s.close()

log.info(f"{len(per_model)} models processed: {sorted(per_model)}")

2026-08-17 16:34:48 | INFO | 07_wrf_hourly_intensity | [access-cm2] 1,051,198 hourly steps -> 120 water-year maxima (1980-09-01 .. 2100-08-31)


2026-08-17 16:34:56 | INFO | 07_wrf_hourly_intensity | [ec-earth3] 1,051,896 hourly steps -> 120 water-year maxima (1980-09-01 .. 2100-08-31)


2026-08-17 16:35:04 | INFO | 07_wrf_hourly_intensity | [miroc6] 1,051,896 hourly steps -> 120 water-year maxima (1980-09-01 .. 2100-08-31)


2026-08-17 16:35:04 | INFO | 07_wrf_hourly_intensity | 3 models processed: ['access-cm2', 'ec-earth3', 'miroc6']


## Ensemble stats, clip, write NetCDF + GeoTIFF + summary

In [4]:
from shapely.geometry import Point
from shapely.prepared import prep
from scipy.interpolate import griddata
from osgeo import gdal, osr
gdal.UseExceptions()

LAT2 = ref["lat"].values   # (y, x) curvilinear
LON2 = ref["lon"].values
prep_b = prep(buffered_ll)
mask2 = np.vectorize(lambda x, y: prep_b.contains(Point(x, y)))(LON2, LAT2)
log.info(f"clip mask: {int(mask2.sum())} of {mask2.size} WRF cells inside buffered boundary")


def write_geotiff_curvilinear(arr2, path_utm, res_deg=0.04):
    """Nearest-resample the curvilinear 9 km field to a regular lat/lon grid, write a
    lat/lon GeoTIFF, then warp to EPSG:26910. res_deg is finer than the native ~0.08
    deg spacing so no cells are dropped; values stay blocky (nearest) - honest 9 km."""
    glon = np.arange(LON2.min(), LON2.max() + res_deg, res_deg)
    glat = np.arange(LAT2.max(), LAT2.min() - res_deg, -res_deg)   # north-up
    gx, gy = np.meshgrid(glon, glat)
    pts = np.column_stack([LON2.ravel(), LAT2.ravel()])
    grid = griddata(pts, arr2.ravel(), (gx, gy), method="nearest")
    grid = np.where(np.isfinite(grid), grid, -9999).astype("float32")
    tmp = str(path_utm) + "._ll.tif"
    drv = gdal.GetDriverByName("GTiff")
    dso = drv.Create(tmp, grid.shape[1], grid.shape[0], 1, gdal.GDT_Float32)
    dso.SetGeoTransform([float(glon[0]) - res_deg / 2, res_deg, 0,
                         float(glat[0]) + res_deg / 2, 0, -res_deg])
    srs = osr.SpatialReference(); srs.ImportFromEPSG(4326)
    dso.SetProjection(srs.ExportToWkt())
    band = dso.GetRasterBand(1)
    band.WriteArray(grid)
    band.SetNoDataValue(-9999)
    dso.FlushCache(); dso = None
    gdal.Warp(str(path_utm), tmp, dstSRS=f"EPSG:{cfg['crs']['target_epsg']}",
              creationOptions=["COMPRESS=DEFLATE"])
    Path(tmp).unlink(missing_ok=True)


def coords_da(arr2):
    return xr.DataArray(arr2, coords={"lat": (("y", "x"), LAT2),
                                      "lon": (("y", "x"), LON2)}, dims=("y", "x"))


ensembles = {}   # (field, scen_tag, hz_tag) -> {stat: 2-D array}
summary = []
tif_dir = PROCESSED / "geotiff"
tif_dir.mkdir(exist_ok=True)

windows = [("max_1hr_precip", "baseline", f"{BASE[0]}-{BASE[1]}", "baseline")] + \
          [("max_1hr_precip", SCEN, hz["name"], hz["name"]) for hz in HORIZONS] + \
          [("cf_1hr_precip", SCEN, hz["name"], "cf_" + hz["name"]) for hz in HORIZONS]

for field, scen_tag, hz_tag, key in windows:
    stack = xr.concat([coords_da(per_model[m][key].values) for m in sorted(per_model)],
                      dim="model")
    ens = {"median": stack.median("model"),
           "p10": stack.quantile(0.10, "model").drop_vars("quantile", errors="ignore"),
           "p90": stack.quantile(0.90, "model").drop_vars("quantile", errors="ignore")}
    ensembles[key] = {s: d.values for s, d in ens.items()}
    for stat, da in ens.items():
        da = da.where(xr.DataArray(mask2, dims=("y", "x")))
        tag = f"{field}_{scen_tag}_{hz_tag.replace('-', '_')}_{stat}"
        da.attrs["units"] = "ratio" if field.startswith("cf") else "mm/hr"
        da.attrs["note"] = ("per-model change factor vs own WY baseline, then ensemble"
                            if field.startswith("cf") else
                            "mean water-year (Sep-Aug) max 1-hr rate; UCLA WRF d02 9 km")
        da.to_dataset(name=field).to_netcdf(PROCESSED / f"{tag}.nc")
        if cfg["run"]["write_geotiff"] and not field.startswith("cf"):
            write_geotiff_curvilinear(da.values, tif_dir / f"{tag}.tif")
        vals = da.values[mask2]
        summary.append({"metric": field, "scenario": scen_tag, "horizon": hz_tag,
                        "stat": stat, "n_models": int(stack.sizes["model"]),
                        "spatial_mean": float(np.nanmean(vals)),
                        "spatial_min": float(np.nanmin(vals)),
                        "spatial_max": float(np.nanmax(vals))})

sm = pd.DataFrame(summary)
sm.to_csv(OUTPUTS / "summary_wrf_i1h.csv", index=False)
log.info(f"wrote {len(sm)} summary rows -> outputs/summary_wrf_i1h.csv; "
         f"NetCDF per stat in data/processed/, GeoTIFF for absolute fields")
sm.round(2)

2026-08-17 16:35:07 | INFO | 07_wrf_hourly_intensity | clip mask: 40 of 210 WRF cells inside buffered boundary


2026-08-17 16:35:09 | INFO | 07_wrf_hourly_intensity | wrote 21 summary rows -> outputs/summary_wrf_i1h.csv; NetCDF per stat in data/processed/, GeoTIFF for absolute fields


,metric,scenario,horizon,stat,n_models,spatial_mean,spatial_min,spatial_max
0,max_1hr_precip,baseline,1981-2010,median,3,9.70,6.68,13.48
1,max_1hr_precip,baseline,1981-2010,p10,3,9.44,6.34,12.96
2,max_1hr_precip,baseline,1981-2010,p90,3,10.07,6.73,13.88
3,max_1hr_precip,ssp370,2020-2049,median,3,10.80,7.11,14.88
4,max_1hr_precip,ssp370,2020-2049,p10,3,10.47,6.97,14.76
5,max_1hr_precip,ssp370,2020-2049,p90,3,11.72,7.98,16.50
6,max_1hr_precip,ssp370,2040-2069,median,3,11.23,7.62,15.98
7,max_1hr_precip,ssp370,2040-2069,p10,3,10.98,7.47,15.60
8,max_1hr_precip,ssp370,2040-2069,p90,3,11.65,7.94,16.01
9,max_1hr_precip,ssp370,2070-2099,median,3,13.02,8.81,19.01


## Cell-polygon GeoJSON for the storm events explorer
One quad per WRF cell (corners = midpoints between neighboring cell centers,
edges extrapolated), cells intersecting the buffered boundary only. Properties are
ensemble medians (+ p10/p90) for the baseline and horizons, plus change factors.
Publication into `html/tahoe-precip-events.html` is manual, after review.

In [5]:
import json
from shapely.geometry import Polygon

def corners(c2):
    """(y, x) cell centers -> (y+1, x+1) corner grid via padding + 4-point means."""
    p = np.pad(c2, 1, mode="edge")
    p[0, :] = 2 * p[1, :] - p[2, :]; p[-1, :] = 2 * p[-2, :] - p[-3, :]
    p[:, 0] = 2 * p[:, 1] - p[:, 2]; p[:, -1] = 2 * p[:, -2] - p[:, -3]
    return 0.25 * (p[:-1, :-1] + p[:-1, 1:] + p[1:, :-1] + p[1:, 1:])

clat, clon = corners(LAT2), corners(LON2)
HZ_TAGS = {hz["name"]: hz["name"][-4:] for hz in HORIZONS}   # '2020-2049' -> '2049'

features = []
ny, nx = LAT2.shape
for j in range(ny):
    for i in range(nx):
        ring = [[round(float(clon[a, b]), 5), round(float(clat[a, b]), 5)]
                for a, b in ((j, i), (j, i + 1), (j + 1, i + 1), (j + 1, i))]
        if not Polygon(ring).intersects(buffered_ll):
            continue
        props = {"cell": f"{j}_{i}",
                 "base": round(float(ensembles["baseline"]["median"][j, i]), 1)}
        for hz in HORIZONS:
            t = HZ_TAGS[hz["name"]]
            props[f"i1h_{t}"] = round(float(ensembles[hz["name"]]["median"][j, i]), 1)
            props[f"lo_{t}"] = round(float(ensembles[hz["name"]]["p10"][j, i]), 1)
            props[f"hi_{t}"] = round(float(ensembles[hz["name"]]["p90"][j, i]), 1)
            props[f"cf_{t}"] = round(float(ensembles["cf_" + hz["name"]]["median"][j, i]), 3)
            props[f"cflo_{t}"] = round(float(ensembles["cf_" + hz["name"]]["p10"][j, i]), 3)
            props[f"cfhi_{t}"] = round(float(ensembles["cf_" + hz["name"]]["p90"][j, i]), 3)
        features.append({"type": "Feature",
                         "geometry": {"type": "Polygon", "coordinates": [ring + [ring[0]]]},
                         "properties": props})

gj = {"type": "FeatureCollection",
      "name": "wrf_i1h_grid",
      "crs": {"type": "name", "properties": {"name": "EPSG:4326"}},
      "features": features}
out = OUTPUTS / "wrf_i1h_grid.geojson"
payload = json.dumps(gj, separators=(",", ":"))
out.write_text(payload)
append_manifest({"file": str(out.relative_to(ROOT)),
                 "source_url": "s3://cadcat/wrf/ucla (hourly prec, 3 models, ssp370)",
                 "size_bytes": out.stat().st_size, "sha256": sha256_file(out),
                 "retrieved_date": str(pd.Timestamp.today().date()),
                 "notebook": "07_wrf_hourly_intensity"})
log.info(f"wrote {out.name}: {len(features)} cells, {len(payload):,} chars")
log.info("To publish: in html/tahoe-precip-events.html replace the object after "
         "'const WRF_I1H = /*__WRF_I1H__*/' with this file's contents "
         "(review the numbers first - the page is public).")

2026-08-17 16:35:09 | INFO | 07_wrf_hourly_intensity | wrote wrf_i1h_grid.geojson: 54 cells, 27,451 chars


2026-08-17 16:35:09 | INFO | 07_wrf_hourly_intensity | To publish: in html/tahoe-precip-events.html replace the object after 'const WRF_I1H = /*__WRF_I1H__*/' with this file's contents (review the numbers first - the page is public).
